Definição da Estrutura

In [0]:
#Bibliotecas Necessárias
from pyspark.sql import functions
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType, DoubleType, DateType, LongType, TimestampType

In [0]:
df = spark.read.table("project_apis.bronze.pix_statistics")

df_schema = StructType([
    StructField("value", ArrayType(
    StructType([
             StructField("AnoMes", StringType(), True),
             StructField("Municipio_Ibge", StringType(), True),
             StructField("Municipio", StringType(), True),
             StructField("Estado_Ibge", StringType(), True),
             StructField("Estado", StringType(), True),
             StructField("Sigla_Regiao", StringType(), True),
             StructField("Regiao", StringType(), True),
             StructField("VL_PagadorPF", StringType(), True),
             StructField("QT_PagadorPF", StringType(), True),
             StructField("VL_PagadorPJ", StringType(), True),
             StructField("QT_PagadorPJ", StringType(), True),
             StructField("VL_RecebedorPF", StringType(), True),
             StructField("QT_RecebedorPF", StringType(), True),
             StructField("VL_RecebedorPJ", StringType(), True),
             StructField("QT_RecebedorPJ", StringType(), True),
             StructField("QT_PES_PagadorPF", StringType(), True),
             StructField("QT_PES_PagadorPJ", StringType(), True),
             StructField("QT_PES_RecebedorPF", StringType(), True),
             StructField("QT_PES_RecebedorPJ", StringType(), True)])))])

print(df_schema)

In [0]:
df1 = df.withColumn("StructValues", functions.from_json(functions.col("raw_json"), df_schema))
#df1.limit(5).show()
df1.printSchema()

In [0]:
len(df.select("raw_json").first()[0])

In [0]:
#Exploding
df2 = df1.withColumn("exploding", functions.explode(functions.col("StructValues.value")))
#df2.limit(5).show()
df2.printSchema()

In [0]:
df_split = df2.select(
    functions.col("exploding.AnoMes").cast(IntegerType()).alias("AnoMes"),
    functions.substring(functions.col("exploding.AnoMes"), 1, 4).cast(IntegerType()).alias("Ano"),
    functions.substring(functions.col("exploding.AnoMes"), 5, 2).cast(IntegerType()).alias("Mes"),
    functions.col("exploding.Municipio_Ibge").cast(StringType()).alias("Municipio_Ibge")
)

df_split.limit(5).show()